> **Data note:** this notebook was run on a private clinical cohort that is not included in this repository, so it will not run as-is. The public diabetes dataset used in the paper is in `data/`; see `03_rail_d.ipynb` and `04_c_rail_d.ipynb` for runnable versions.

In [ ]:
from dotenv import load_dotenv, find_dotenv
import requests
import os
import json
import math
import argparse
import pandas as pd
from typing import List, Literal
from pydantic import BaseModel, Field, ValidationError
from langchain.output_parsers import PydanticOutputParser

from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

from tqdm import tqdm

from typing import List, Literal, Optional
from pydantic import BaseModel, Field

In [ ]:
EXCEL_PATH = "../data/PatientsF_with_slopes_new.xlsx"
SHEET = 0                                           # or a sheet name
MODEL = "gpt-4o-mini"   # any OpenAI chat model
TEMPERATURE = 0.0                                   # deterministic classification
TARGET_COL = "IEC_HS"                               # ground-truth label in your file
OUT_CSV = "../data/iec_hs_predictions.csv"
max_tokens=768

In [ ]:
# OpenAI API access: put OPENAI_API_KEY in your environment or in a local .env file
# (see .env.example). Never commit your key.
load_dotenv(find_dotenv())
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set")

In [ ]:
"""
class IecHsRationale(BaseModel):
    iec_hs_true: Literal[0, 1] = Field(..., description="True IEC_HS label from the dataset (0 or 1).")
    verdict_match: bool = Field(..., description="Whether the reasoning supports the given true label.")
    key_supporting_factors: List[str] = Field(..., description="Bulleted factors supporting the true label.")
    key_contradicting_factors: List[str] = Field(..., description="Bulleted factors that go against the true label.")
    short_explanation: str = Field(..., description="2-4 sentences explaining why IEC_HS has this value for the patient.")
    confidence: float = Field(..., ge=0.0, le=1.0, description="Self‑reported confidence 0–1.")

 
"""

class IecHsRationale(BaseModel):
    iec_hs_true: Literal[0, 1] = Field(..., description="True IEC_HS label from the dataset (0 or 1).")
    verdict_match: bool = Field(..., description="Whether the reasoning supports the given true label.")

    #iec_hs_true: Optional[int] = Field(None, description="True IEC_HS label (0 or 1). Accepts truthy/falsy forms.")
    #verdict_match: Optional[bool] = Field( None, description="Whether the reasoning supports the given true label.")

    key_supporting_factors: List[str] = Field(
        default_factory=list, 
        description="Bulleted factors supporting the true label."
    )
    key_contradicting_factors: List[str] = Field(
        default_factory=list, 
        description="Bulleted factors that go against the true label."
    )
    short_explanation: Optional[str] = Field(
        None, 
        description="2-4 sentences explaining why IEC_HS has this value for the patient."
    )
    confidence: Optional[float] = Field(
        None, ge=0.0, le=1.0, 
        description="Self-reported confidence 0–1."
    )


parser = PydanticOutputParser(pydantic_object=IecHsRationale)

SYSTEM_MSG = """You are a careful clinical explainer.
Given an anonymized patient row, explain *why the true IEC_HS label* is what it is,
using only the provided fields. Do not invent data.
Keep the explanation clinically plausible and concise.
Return ONLY valid JSON that matches the provided schema."""

HUMAN_TMPL = """
Patient row (JSON key→value):
{patient_features_json}

True label:
IEC_HS = {iec_hs_true}

Instructions:
1) Explain why the TRUE IEC_HS could be {iec_hs_true} using the other fields (e.g., ferritin, CRP, Hb/Platelet/WBC/ANC,
   triglycerides, fibrinogen, LDH, bulky disease, CRS, tocilizumab, etc., when available).
2) If evidence is mixed, tell why the prediction should still be {iec_hs_true}.
3) Return ONLY JSON, no extra text.

Use this format:
{format_instructions}
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", SYSTEM_MSG), ("human", HUMAN_TMPL)]
)

# IMPORTANT: inject the schema instructions via variable
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

In [ ]:
llm = ChatOpenAI(
        model=MODEL,
        temperature=0,
        max_retries=2,
        max_tokens=1024,
        api_key=api_key,
    )

chain = prompt | llm | parser

In [ ]:
def row_to_features(row: pd.Series, target_col: str = "IEC_HS"):
    d = {}
    for k, v in row.items():
        if k == target_col:
            continue
        # normalize NaNs
        if isinstance(v, float) and math.isnan(v):
            d[k] = None
        else:
            try:
                d[k] = v.item() if hasattr(v, "item") else v
            except Exception:
                d[k] = v
    return d

In [ ]:
def row_to_features(row: pd.Series) -> dict:
    """Convert a row to a JSON-serializable dict, handling NaN/None cleanly."""
    out = {}
    for k in feature_cols:
        v = row[k]
        if isinstance(v, float) and math.isnan(v):
            out[k] = None
        else:
            # cast numpy types to native
            if hasattr(v, "item"):
                try:
                    v = v.item()
                except Exception:
                    pass
            out[k] = v
    return out


# Load data
df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET)

# Build feature set: everything except the target
feature_cols = [c for c in df.columns if c != TARGET_COL]

In [ ]:
results = []

errored_rows = []  # to keep track of rows where exceptions happened

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Scoring patients"):
    try:
        patient_features_json = row_to_features(row)  # same as before (target excluded)
        assert TARGET_COL not in patient_features_json
        iec_true = int(row[TARGET_COL])

        pred: IecHsRationale = chain.invoke({
            "patient_features_json": json.dumps(patient_features_json, ensure_ascii=False),
            "iec_hs_true": str(iec_true),
        })

        print(pred.short_explanation)

        d = pred.model_dump()
        export_row = row.to_dict()  # start with ALL original columns for this row
        export_row["iec_hs_true"] = d["iec_hs_true"]
        export_row["verdict_match"] = d["verdict_match"]
        export_row["key_supporting_factors"] = " • ".join(d.get("key_supporting_factors", []))
        export_row["key_contradicting_factors"] = " • ".join(d.get("key_contradicting_factors", []))
        export_row["short_explanation"] = d["short_explanation"]
        export_row["confidence"] = d["confidence"]

        results.append(export_row)

    except Exception as e:
        errored_rows.append((idx, str(e)))  # store row index + error message
        continue  # skip to next row

"""
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Scoring patients"):
    patient_features_json = row_to_features(row)  # same as before (target excluded)
    assert TARGET_COL not in patient_features_json
    iec_true = int(row[TARGET_COL])

    pred: IecHsRationale = chain.invoke({
        "patient_features_json": json.dumps(patient_features_json, ensure_ascii=False),
        "iec_hs_true": str(iec_true),
    })
    print(pred.short_explanation)

    d = pred.model_dump()
    # start with ALL original columns for this row
    export_row = row.to_dict()
    # append new columns
    export_row["iec_hs_true"] = d["iec_hs_true"]
    export_row["verdict_match"] = d["verdict_match"]
    export_row["key_supporting_factors"] = " • ".join(d.get("key_supporting_factors", []))
    export_row["key_contradicting_factors"] = " • ".join(d.get("key_contradicting_factors", []))
    export_row["short_explanation"] = d["short_explanation"]
    export_row["confidence"] = d["confidence"]

    results.append(export_row)
"""
# ensure original columns first, then new ones
new_cols = [
    "iec_hs_true", "verdict_match",
    "key_supporting_factors", "key_contradicting_factors",
    "short_explanation", "confidence",
]
ordered_cols = list(df.columns) + [c for c in new_cols if c not in df.columns]

out_df = pd.DataFrame(results)[ordered_cols]
out_df.to_excel("../data/iec_hs_rationales.xlsx", index=False)
print("Saved → iec_hs_rationales.xlsx")
# After loop:
if errored_rows:
    print(f"\n{len(errored_rows)} rows failed during processing:")
    for row_idx, err in errored_rows:
        print(f"Row {row_idx}: {err}")